# Question Answering (QA)

## What is QA?
Given a question and optionally a context passage, produce the correct answer.

**Types:**
- **Extractive QA**: Answer is a span extracted from context (SQuAD-style)
- **Generative QA**: Answer is generated (not necessarily in context)
- **Open-domain QA**: No fixed context retrieve from large corpus
- **Visual QA**: Answer questions about images
- **Knowledge-intensive QA**: Requires factual world knowledge

---

## 1. Extractive QA

### SQuAD Format
Given: context passage + question → predict start and end positions of answer span.

### BERT for Extractive QA

Input: `[CLS] Question [SEP] Context [SEP]`

Two classification heads on top of BERT output:

$$P(\text{start}_i) = \text{softmax}(W_S \cdot h_i)$$
$$P(\text{end}_i) = \text{softmax}(W_E \cdot h_i)$$

Loss:
$$L = -\frac{1}{N}\sum_i [\log P(\text{start}_{y_i^s}) + \log P(\text{end}_{y_i^e})]$$

Prediction: $\arg\max_{i \leq j} P(\text{start}_i) \cdot P(\text{end}_j)$

---

## 2. Open-Domain QA

### Retrieval-Augmented QA (DPR + Reader)

**DPR (Dense Passage Retrieval)**:
1. **Retriever**: Encode question $q$ and passages $p$ with dual encoder
   $$\text{sim}(q, p) = E_Q(q)^T \cdot E_P(p)$$
2. **Reader**: Re-rank and extract answer from top-k passages

Training: Maximize similarity for positive passage, minimize for negatives (in-batch negatives).

---

## 3. Evaluation

### Exact Match (EM):
$$EM = \frac{1}{N} \sum_i \mathbf{1}[\text{prediction}_i = \text{gold}_i]$$

### Token-level F1:
$$F1 = \frac{2 \cdot |\text{pred} \cap \text{gold}|}{|\text{pred}| + |\text{gold}|}$$

Tokens are matched after normalization (lowercase, remove articles/punctuation).

In [1]:
# ============================================================
# EXTRACTIVE QA WITH HUGGING FACE
# ============================================================
try:
    from transformers import pipeline
    
    qa_pipeline = pipeline(
        "question-answering",
        model="deepset/roberta-base-squad2"
    )
    
    context = """
    The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
    It is named after the engineer Gustave Eiffel, whose company designed and built the tower.
    Constructed from 1887 to 1889 as the centerpiece of the 1889 World's Fair, it was initially
    criticized by some of France's leading artists and intellectuals for its design, but it has
    become a global cultural icon of France and one of the most recognizable structures in the world.
    The Eiffel Tower is the most-visited paid monument in the world; 6.91 million people ascended it in 2015.
    """
    
    questions = [
        "Where is the Eiffel Tower located?",
        "Who designed the Eiffel Tower?",
        "When was the Eiffel Tower built?",
        "How many people visited in 2015?"
    ]
    
    print("Extractive QA Results:")
    print("=" * 60)
    for question in questions:
        result = qa_pipeline(question=question, context=context)
        print(f"Q: {question}")
        print(f"A: {result['answer']} (score: {result['score']:.3f})")
        print()

except Exception as e:
    print(f"Requires model download: {e}")

Requires model download: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"


In [2]:
# ============================================================
# QA EVALUATION METRICS FROM SCRATCH
# ============================================================
import string
import re
from collections import Counter

def normalize_answer(s):
    """Normalize answer: lowercase, remove articles, punctuation, extra spaces"""
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def remove_punc(text):
        return ''.join(c for c in text if c not in set(string.punctuation))
    def white_space_fix(text):
        return ' '.join(text.split())
    return white_space_fix(remove_articles(remove_punc(s.lower())))

def exact_match(prediction, gold):
    return normalize_answer(prediction) == normalize_answer(gold)

def f1_score(prediction, gold):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

# Test
test_cases = [
    ("Paris, France", "Paris, France", "Perfect match"),
    ("the Eiffel Tower", "Eiffel Tower", "Article difference"),
    ("Gustave Eiffel", "Gustave Alexandre Eiffel", "Partial match"),
    ("1887", "1889", "Wrong answer"),
]

print(f"{'Case':<30} {'EM':<6} {'F1':<6}")
print("-" * 45)
for pred, gold, name in test_cases:
    em = exact_match(pred, gold)
    f1 = f1_score(pred, gold)
    print(f"{name:<30} {str(em):<6} {f1:.3f}")

Case                           EM     F1    
---------------------------------------------
Perfect match                  True   1.000
Article difference             True   1.000
Partial match                  False  0.800
Wrong answer                   False  0.000


In [3]:
# ============================================================
# FINE-TUNING ON SQuAD FORMAT
# ============================================================
print("Fine-tuning BERT on SQuAD:")
print()
print("""
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from transformers import TrainingArguments, Trainer
from datasets import load_dataset

# Load SQuAD dataset
dataset = load_dataset('squad')

# Load model
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Preprocessing function
def preprocess_function(examples):
    inputs = tokenizer(
        examples['question'],
        examples['context'],
        max_length=384,
        truncation='only_second',
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length'
    )
    # Find start/end positions in tokenized input
    # ... (offset mapping logic)
    return inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

training_args = TrainingArguments(
    output_dir='./qa_model',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=3e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
)
trainer.train()
""")

Fine-tuning BERT on SQuAD:


from transformers import AutoTokenizer, AutoModelForQuestionAnswering
from transformers import TrainingArguments, Trainer
from datasets import load_dataset

# Load SQuAD dataset
dataset = load_dataset('squad')

# Load model
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Preprocessing function
def preprocess_function(examples):
    inputs = tokenizer(
        examples['question'],
        examples['context'],
        max_length=384,
        truncation='only_second',
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length'
    )
    # Find start/end positions in tokenized input
    # ... (offset mapping logic)
    return inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True)

training_args = TrainingArguments(
    output_dir='./qa_model',
    num_train_epochs=3,
  

In [4]:
# ============================================================
# GENERATIVE QA WITH LLM
# ============================================================
try:
    from transformers import pipeline
    
    # T5 for abstractive QA
    qa_gen = pipeline("text2text-generation", model="google/flan-t5-small")
    
    questions = [
        "What is the capital of France?",
        "Who wrote Romeo and Juliet?",
        "What is the boiling point of water in Celsius?"
    ]
    
    print("Generative QA (Flan-T5):")
    for q in questions:
        result = qa_gen(q, max_new_tokens=50)
        print(f"Q: {q}")
        print(f"A: {result[0]['generated_text']}")
        print()
        
except Exception as e:
    print(f"Requires model download: {e}")

Requires model download: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"


## Additional Learning Resources

### Papers
- [SQuAD: 100,000+ Questions for Machine Comprehension](https://arxiv.org/abs/1606.05250) Rajpurkar et al., 2016
- [SQuAD 2.0: Know What You Don't Know](https://arxiv.org/abs/1806.03822) Rajpurkar et al., 2018
- [DPR: Dense Passage Retrieval for Open-Domain QA](https://arxiv.org/abs/2004.04906) Karpukhin et al., 2020
- [RAG: Retrieval-Augmented Generation for Knowledge-Intensive NLP](https://arxiv.org/abs/2005.11401) Lewis et al., 2020
- [FusionNet / Bidirectional Attention Flow (BiDAF)](https://arxiv.org/abs/1611.01603)

### Datasets
- [SQuAD v1.1 and v2.0](https://rajpurkar.github.io/SQuAD-explorer/)
- [TriviaQA](https://nlp.cs.washington.edu/triviaqa/)
- [Natural Questions](https://ai.google.com/research/NaturalQuestions)
- [HotpotQA](https://hotpotqa.github.io/) Multi-hop reasoning

### Tutorials
- [Hugging Face QA Guide](https://huggingface.co/docs/transformers/tasks/question_answering)
- [Hugging Face QA Course Chapter 7](https://huggingface.co/learn/nlp-course/chapter7/7)